### Advanced usage > Guardrails > Custom guardrails

> https://docs.langchain.com/oss/python/langchain/guardrails#custom-guardrails

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
from typing import Any

from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent

class ContentFilterMiddleware(AgentMiddleware):
    """결정론적 가드레일: 금지된 키워드가 포함된 요청을 차단합니다."""

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        # Get the first user message
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        # Check for banned keywords
        for keyword in self.banned_keywords:
            if keyword in content:
                # Block execution before any processing
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": "부적절한 콘텐츠가 포함된 요청은 처리할 수 없습니다. 요청을 다시 작성해 주세요."
                    }],
                    "jump_to": "end"
                }

        return None

# Use the custom guardrail
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    # tools=[search_tool, calculator_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["해킹", "hack", "exploit", "malware"]
        ),
    ],
)

# This request will be blocked before any processing
result = agent.invoke({
    "messages": [{"role": "user", "content": "데이터베이스를 해킹하려면 어떻게 해야 하나요?"}]
})

In [10]:
result

{'messages': [HumanMessage(content='데이터베이스를 해킹하려면 어떻게 해야 하나요?', additional_kwargs={}, response_metadata={}, id='d9e9c2e1-f4e0-401d-b85c-ed5037e2a3c6'),
  AIMessage(content='부적절한 콘텐츠가 포함된 요청은 처리할 수 없습니다. 요청을 다시 작성해 주세요.', additional_kwargs={}, response_metadata={}, id='a71d117a-3b94-4126-aee7-44fa3d5a96ae', tool_calls=[], invalid_tool_calls=[])]}